In [ ]:
#  les fichiers présents dans Bronze
files = notebookutils.fs.ls("Files/bronze")

for file in files:
    print(file.name, file.size, "bytes")


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 7, Finished, Available, Finished, False)

WA_Fn-UseC_-Telco-Customer-Churn.csv 977501 bytes


In [ ]:

from pyspark.sql import functions as F

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv("Files/bronze/")
)


print("Nombre de lignes :", df.count())
print("Nombre de colonnes :", len(df.columns))

df.printSchema()
display(df.limit(10))


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 10, Finished, Available, Finished, False)

Nombre de lignes : 7043
Nombre de colonnes : 21
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 62e95538-17c7-4d52-bc4e-d7abdf2e9219)

In [ ]:

print("Nombre de clients distincts :",
      df.select("customerID").distinct().count())

print("Nombre de lignes dupliquées :",
      df.count() - df.dropDuplicates().count())


null_counts = df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(null_counts)


display(df.groupBy("Churn").count())


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 9, Finished, Available, Finished, False)

Nombre de clients distincts : 7043
Nombre de lignes dupliquées : 0


SynapseWidget(Synapse.DataFrame, deccd432-35bd-4c78-b883-770bd5371a79)

SynapseWidget(Synapse.DataFrame, e9ecce47-ea10-4fb4-9390-26abdb772b4e)

In [ ]:

from pyspark.sql import functions as F

# A. Vérifier les doublons
print("Lignes identiques en double :",
      df.count() - df.dropDuplicates().count())

print("CustomerID distincts :",
      df.select("customerID").distinct().count())

print("CustomerID nulls :",
      df.filter(
          F.col("customerID").isNull()
      ).count())

# B. Vérifier les valeurs manquantes 
quality_checks = df.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(quality_checks)


display(
    df.filter(
        F.col("TotalCharges").isNull() |
        (F.trim(F.col("TotalCharges")) == "") |
        F.col("TotalCharges").cast("double").isNull()
    ).select("customerID", "tenure",
             "MonthlyCharges", "TotalCharges", "Churn")
)

# D. Vérifier les valeurs de la cible
display(df.groupBy("Churn").count())


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 11, Finished, Available, Finished, False)

Lignes identiques en double : 0
CustomerID distincts : 7043
CustomerID nulls : 0


SynapseWidget(Synapse.DataFrame, d9df4b05-c3d6-46aa-89f4-56d83d3ca97b)

SynapseWidget(Synapse.DataFrame, fda47e4e-22c9-4a03-9aa2-71198e03b682)

SynapseWidget(Synapse.DataFrame, 69294588-a95f-42f9-b521-121a038c48ec)

In [ ]:

from pyspark.sql import functions as F

# 1. Vérifier les valeurs de Churn
display(df.groupBy("Churn").count())

# 2. Vérifier les valeurs non numériques de TotalCharges
display(
    df.filter(
        F.col("TotalCharges").cast("double").isNull()
    ).select(
        "customerID", "tenure",
        "MonthlyCharges", "TotalCharges", "Churn"
    )
)

# 3. Vérifier les chaînes vides après suppression des espaces
print(
    "TotalCharges vides :",
    df.filter(
        F.trim(F.col("TotalCharges")) == ""
    ).count()
)

# 4. Vérifier les valeurs négatives 
print(
    "Tenure négative :",
    df.filter(F.col("tenure") < 0).count()
)

print(
    "MonthlyCharges négatif :",
    df.filter(F.col("MonthlyCharges") < 0).count()
)


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f40650f6-9c90-4ba5-97c4-00df3d5d67a9)

SynapseWidget(Synapse.DataFrame, 9661aa92-2548-4700-a360-fed58d0fc427)

TotalCharges vides : 11
Tenure négative : 0
MonthlyCharges négatif : 0


In [11]:

from pyspark.sql import functions as F

empty_total = df.filter(
    F.trim(F.col("TotalCharges")) == ""
)

display(
    empty_total.select(
        "customerID",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Contract",
        "Churn"
    )
)

print(
    "Lignes vides avec tenure = 0 :",
    empty_total.filter(F.col("tenure") == 0).count()
)


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e3fed80c-5899-4d23-9ad8-23a815f1dee7)

Lignes vides avec tenure = 0 : 11


In [ ]:

from pyspark.sql import functions as F

# 1. Normaliser les chaînes vides de TotalCharges
df_clean = df.withColumn(
    "TotalCharges",
    F.when(
        F.trim(F.col("TotalCharges")) == "",
        F.lit(None).cast("string")
    ).otherwise(F.trim(F.col("TotalCharges")))
)

# 2. Convertir TotalCharges en nombre
df_clean = df_clean.withColumn(
    "TotalCharges",
    F.col("TotalCharges").cast("double")
)


df_clean = df_clean.withColumn(
    "TotalCharges",
    F.when(
        (F.col("tenure") == 0) &
        F.col("TotalCharges").isNull(),
        F.lit(0.0)
    ).otherwise(F.col("TotalCharges"))
)

# 4. Identifier les lignes rejetées
invalid_condition = (
    F.col("customerID").isNull() |
    (F.trim(F.col("customerID")) == "") |
    F.col("tenure").isNull() |
    (F.col("tenure") < 0) |
    F.col("MonthlyCharges").isNull() |
    (F.col("MonthlyCharges") < 0) |
    F.col("TotalCharges").isNull() |
    (F.col("TotalCharges") < 0) |
    (~F.col("Churn").isin("Yes", "No"))
)

df_rejected = df_clean.filter(invalid_condition)
df_silver = df_clean.filter(~invalid_condition)

print("Lignes source :", df.count())
print("Lignes Silver :", df_silver.count())
print("Lignes rejetées :", df_rejected.count())

df_silver.printSchema()


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 14, Finished, Available, Finished, False)

Lignes source : 7043
Lignes Silver : 7043
Lignes rejetées : 0
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [ ]:


print(
    "TotalCharges encore null :",
    df_silver.filter(
        F.col("TotalCharges").isNull()
    ).count()
)

print(
    "Clients avec tenure = 0 et TotalCharges = 0 :",
    df_silver.filter(
        (F.col("tenure") == 0) &
        (F.col("TotalCharges") == 0)
    ).count()
)

display(
    df_silver.groupBy("Churn").count()
)


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 15, Finished, Available, Finished, False)

TotalCharges encore null : 0
Clients avec tenure = 0 et TotalCharges = 0 : 11


SynapseWidget(Synapse.DataFrame, 3ab8e36f-9cdd-42b9-8022-b74756eda0a1)

In [14]:

# Enregistrer la table Silver dans le Lakehouse
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_telecom_customers")
)

print("Table silver_telecom_customers créée.")


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 16, Finished, Available, Finished, False)

Table silver_telecom_customers créée.


In [15]:

silver = spark.table("silver_telecom_customers")

print("Nombre de lignes :", silver.count())
print("Nombre de clients uniques :",
      silver.select("customerID").distinct().count())

display(silver.limit(10))


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 17, Finished, Available, Finished, False)

Nombre de lignes : 7043
Nombre de clients uniques : 7043


SynapseWidget(Synapse.DataFrame, 053f4f45-0bec-49a8-9279-1bea6821d218)

In [16]:

# Vérifier les catégories de churn
display(silver.groupBy("Churn").count())

# Vérifier les contrats
display(silver.groupBy("Contract").count())

# Vérifier les valeurs numériques extrêmes
display(
    silver.select(
        F.min("tenure").alias("min_tenure"),
        F.max("tenure").alias("max_tenure"),
        F.min("MonthlyCharges").alias("min_monthly_charges"),
        F.max("MonthlyCharges").alias("max_monthly_charges"),
        F.min("TotalCharges").alias("min_total_charges"),
        F.max("TotalCharges").alias("max_total_charges")
    )
)


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd2070b3-7b11-4065-92e1-932f1751e7ba)

SynapseWidget(Synapse.DataFrame, 930705d1-6956-4cd4-ac7a-eeb313f73aaa)

SynapseWidget(Synapse.DataFrame, 4314bf43-9160-4fad-9a6b-4b166e87b727)

In [17]:

from pyspark.sql import functions as F

silver = spark.table("silver_telecom_customers")

# 1. Vérifier les identifiants clients
print("Nombre de lignes :", silver.count())
print(
    "Nombre de customerID distincts :",
    silver.select("customerID").distinct().count()
)

# 2. Vérifier les catégories de Churn
display(silver.groupBy("Churn").count())

# 3. Vérifier les catégories de contrat
display(silver.groupBy("Contract").count())

# 4. Vérifier les incohérences principales
print(
    "Tenure hors plage [0, 72] :",
    silver.filter(
        (F.col("tenure") < 0) | (F.col("tenure") > 72)
    ).count()
)

print(
    "TotalCharges négatif :",
    silver.filter(F.col("TotalCharges") < 0).count()
)

print(
    "Churn invalide :",
    silver.filter(~F.col("Churn").isin("Yes", "No")).count()
)


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 19, Finished, Available, Finished, False)

Nombre de lignes : 7043
Nombre de customerID distincts : 7043


SynapseWidget(Synapse.DataFrame, 6d1bb4a7-82a5-4d94-8184-b91fe72ea8a9)

SynapseWidget(Synapse.DataFrame, fa6166fa-ed4e-4379-9476-972a983122a7)

Tenure hors plage [0, 72] : 0
TotalCharges négatif : 0
Churn invalide : 0


In [18]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table("silver_telecom_customers")

# Dimension Client : attributs du client et services souscrits
customer_columns = [
    "customerID", "gender", "SeniorCitizen",
    "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies",
    "PaperlessBilling", "PaymentMethod"
]

dim_customer = (
    silver.select(*customer_columns)
    .dropDuplicates(["customerID"])
    .withColumn(
        "customer_key",
        F.row_number().over(
            Window.orderBy("customerID")
        )
    )
)

# Dimension Contrat : types de contrats distincts
dim_contract = (
    silver.select("Contract")
    .distinct()
    .withColumn(
        "contract_key",
        F.row_number().over(
            Window.orderBy("Contract")
        )
    )
)

# Table de faits : une ligne par client dans cet instantané
fact = (
    silver
    .join(
        dim_customer.select("customerID", "customer_key"),
        on="customerID",
        how="inner"
    )
    .join(dim_contract, on="Contract", how="inner")
    .select(
        "customer_key",
        "contract_key",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Churn"
    )
)

# Enregistrer les tables au format Delta
dim_customer.write.format("delta").mode("overwrite").saveAsTable(
    "gold_dim_customer"
)

dim_contract.write.format("delta").mode("overwrite").saveAsTable(
    "gold_dim_contract"
)

fact.write.format("delta").mode("overwrite").saveAsTable(
    "gold_fact_customer_churn"
)

print("Tables Gold créées.")


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 20, Finished, Available, Finished, False)

Tables Gold créées.


In [19]:

fact_gold = spark.table("gold_fact_customer_churn")
customer_gold = spark.table("gold_dim_customer")
contract_gold = spark.table("gold_dim_contract")

print("Lignes Fact :", fact_gold.count())
print("Lignes DimCustomer :", customer_gold.count())
print("Lignes DimContract :", contract_gold.count())

display(fact_gold.groupBy("Churn").count())
display(fact_gold.groupBy("contract_key").count())


StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 21, Finished, Available, Finished, False)

Lignes Fact : 7043
Lignes DimCustomer : 7043
Lignes DimContract : 3


SynapseWidget(Synapse.DataFrame, e6b5aa61-1fb8-405b-b918-0f2284d1effe)

SynapseWidget(Synapse.DataFrame, b3bda1f1-385d-4bb1-897c-ea0a0bce6be9)

In [20]:
from pyspark.sql import functions as F

fact = spark.table("gold_fact_customer_churn")
dim_customer = spark.table("gold_dim_customer")
dim_contract = spark.table("gold_dim_contract")

print("Fact sans doublons de clé client :",
      fact.select("customer_key").distinct().count())

print("Clés client nulles :",
      fact.filter(F.col("customer_key").isNull()).count())

print("Clients absents de DimCustomer :",
      fact.join(
          dim_customer.select("customer_key"),
          on="customer_key",
          how="left_anti"
      ).count())

print("Contrats absents de DimContract :",
      fact.join(
          dim_contract.select("contract_key"),
          on="contract_key",
          how="left_anti"
      ).count())

StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 22, Finished, Available, Finished, False)

Fact sans doublons de clé client : 7043
Clés client nulles : 0
Clients absents de DimCustomer : 0
Contrats absents de DimContract : 0


In [21]:
from pyspark.sql import functions as F

fact = spark.table("gold_fact_customer_churn")
dim_contract = spark.table("gold_dim_contract")

churn_by_contract = (
    fact
    .join(dim_contract, on="contract_key", how="inner")
    .groupBy("Contract")
    .agg(
        F.count("*").alias("TotalCustomers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("ChurnedCustomers")
    )
    .withColumn(
        "ChurnRate",
        F.round(
            F.col("ChurnedCustomers") / F.col("TotalCustomers") * 100,
            2
        )
    )
    .orderBy(F.desc("ChurnRate"))
)

display(churn_by_contract)

StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f8c90815-a59e-448c-b296-91a79c5d8570)

In [22]:
from pyspark.sql import functions as F

fact = spark.table("gold_fact_customer_churn")

churn_kpis = fact.agg(
    F.count("*").alias("TotalCustomers"),
    F.sum(
        F.when(F.col("Churn") == "Yes", 1).otherwise(0)
    ).alias("ChurnedCustomers"),
    F.round(
        F.avg(
            F.when(F.col("Churn") == "Yes", 1.0).otherwise(0.0)
        ) * 100,
        2
    ).alias("OverallChurnRate")
)

display(churn_kpis)

StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4c8c5cd-021e-4d53-b718-b8c72e39b7a7)

In [23]:
from pyspark.sql import functions as F

fact = spark.table("gold_fact_customer_churn")

churn_by_tenure = (
    fact
    .withColumn(
        "TenureGroup",
        F.when(F.col("tenure") <= 12, "0-12 months")
         .when(F.col("tenure") <= 24, "13-24 months")
         .when(F.col("tenure") <= 48, "25-48 months")
         .otherwise("49-72 months")
    )
    .groupBy("TenureGroup")
    .agg(
        F.count("*").alias("TotalCustomers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("ChurnedCustomers"),
        F.round(
            F.avg(
                F.when(F.col("Churn") == "Yes", 1.0)
                 .otherwise(0.0)
            ) * 100,
            2
        ).alias("ChurnRate")
    )
    .orderBy("TenureGroup")
)

display(churn_by_tenure)

StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 43b906fb-0ef1-4ca7-b88c-c2cfcee4cbde)

In [24]:
from pyspark.sql import functions as F

fact = spark.table("gold_fact_customer_churn")
dim_customer = spark.table("gold_dim_customer")

display(
    fact.groupBy("Churn")
    .count()
    .orderBy("Churn")
)

display(
    dim_customer.groupBy("InternetService")
    .agg(
        F.count("*").alias("TotalCustomers")
    )
    .orderBy(F.desc("TotalCustomers"))
)

display(
    dim_customer.groupBy("gender")
    .agg(
        F.count("*").alias("TotalCustomers")
    )
    .orderBy("gender")
)

StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c58ca11-d303-4144-9e09-faf1cdb926d8)

SynapseWidget(Synapse.DataFrame, 5f4db9a5-8892-49d1-883e-2dd26bd08a71)

SynapseWidget(Synapse.DataFrame, c23f02e7-3d9f-4e25-b390-643f1ef81c6e)

In [25]:
from pyspark.sql import functions as F

fact = spark.table("gold_fact_customer_churn")
dim_customer = spark.table("gold_dim_customer")

analysis_df = fact.join(
    dim_customer.select("customer_key", "InternetService", "gender"),
    on="customer_key",
    how="inner"
)

def calculate_churn(group_column):
    return (
        analysis_df
        .groupBy(group_column)
        .agg(
            F.count("*").alias("TotalCustomers"),
            F.sum(
                F.when(F.col("Churn") == "Yes", 1).otherwise(0)
            ).alias("ChurnedCustomers"),
            F.round(
                F.avg(
                    F.when(F.col("Churn") == "Yes", 1.0)
                    .otherwise(0.0)
                ) * 100,
                2
            ).alias("ChurnRate")
        )
        .orderBy(F.desc("ChurnRate"))
    )

display(calculate_churn("InternetService"))
display(calculate_churn("gender"))

StatementMeta(, 078b9f5a-bfba-4793-8c82-950caad59962, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, af25b657-7bf8-401b-88d2-0a61f323f228)

SynapseWidget(Synapse.DataFrame, 2c729dd2-a2ff-4016-9507-12b6f638efc9)

In [1]:
print("=== DIM CONTRACT ===")
spark.table("gold_dim_contract").printSchema()

print("=== FACT CHURN ===")
spark.table("gold_fact_customer_churn").printSchema()

print("=== SILVER CUSTOMERS ===")
spark.table("silver_telecom_customers").printSchema()

StatementMeta(, 181dff00-f9f0-4207-9f11-e8b6c38a3b27, 3, Finished, Available, Finished, False)

=== DIM CONTRACT ===
root
 |-- Contract: string (nullable = true)
 |-- contract_key: integer (nullable = true)

=== FACT CHURN ===
root
 |-- customer_key: integer (nullable = true)
 |-- contract_key: integer (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)

=== SILVER CUSTOMERS ===
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- Streaming

In [1]:
spark.sql("""
SELECT InternetService, COUNT(*) AS TotalCustomers
FROM gold_fact_customer_churn f
JOIN gold_dim_customer d
ON f.customer_key = d.customer_key
GROUP BY InternetService
ORDER BY InternetService
""").show()

StatementMeta(, ba90703b-d791-4a52-984a-8a563b5ff0d1, 3, Finished, Available, Finished, False)

+---------------+--------------+
|InternetService|TotalCustomers|
+---------------+--------------+
|            DSL|          2421|
|    Fiber optic|          3096|
|             No|          1526|
+---------------+--------------+

